
# 00 — Sample 1,500 questions from PopQA

Runs **on the laptop**. Outputs `data/halt_rag_audit_1500.csv`, the input to
`notebooks/02b_wiki2023_build.ipynb` for the full study.

**Strategy**

* Start from the 300 pilot questions (`halt_rag_audit_v5.csv`).
* Apply the band rules **strictly** using current PopQA `s_pop` values:
  tail `s_pop < 100`, head `s_pop > 500`.
* Pilot questions that now violate their band's rule, and pilot questions from a
  duplicate entity, are **dropped and replaced** with new questions from the same band.
* Each remaining slot is filled with a new entity — one question per subject entity
  across the full 1,500.
* Final file: exactly 750 head + 750 tail, 1,500 unique entities, all band rules
  satisfied.

**Why `s_pop < 100` instead of `< 50`?**
Only 356 distinct entities have `s_pop < 50` in PopQA. The threshold was relaxed to
`< 100` (bottom ~10% of PopQA, 1,367 unique entities). The methodology chapter says
`≤ 50`; the final report must note this change and the reason.

**PopQA `s_pop` values changed since the pilot** — four pilot tail questions now have
`s_pop > 50`, and one pilot head question now has `s_pop ≤ 500`. These are dropped and
replaced rather than kept, so the full study applies one consistent standard to all
1,500 questions.

**Output columns** (minimum set for `02b_wiki2023_build.ipynb`):

`question`, `answers_list`, `band`, `wiki_title`, `s_uri`, `s_pop`, `id`, `prop`


## Cell 1 — Setup

In [1]:
import os, sys, ast, json
import pandas as pd
import numpy as np

# repo root is one level above notebooks/
REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path
cfg  = load_config(os.path.join(REPO, "config.yaml"))

SEED         = cfg["seed"]
TAIL_MAX     = cfg["questions"]["tail_max_pop"]   # 100  (full study)
HEAD_MIN     = cfg["questions"]["head_min_pop"]   # 500
N_PER_BAND   = 750                                # target per band for the full study
PILOT_CSV    = data_path(cfg, "audit_v5")         # halt_rag_audit_v5.csv
AUDIT_W23    = data_path(cfg, "audit_w23")        # halt_rag_audit_w23.csv (has s_uri)
OUT_CSV      = data_path(cfg, "audit_1500")       # halt_rag_audit_1500.csv

for p in (PILOT_CSV, AUDIT_W23):
    assert os.path.exists(p), f"Not found: {p}"

print(f"seed={SEED} | tail s_pop < {TAIL_MAX} | head s_pop > {HEAD_MIN} | target {N_PER_BAND} per band")
print(f"output -> {OUT_CSV}")

seed=42 | tail s_pop < 100 | head s_pop > 500 | target 750 per band
output -> E:\shaithu\4th Year\IRP\halt-rag-repo\HALT-RAG\data\halt_rag_audit_1500.csv


## Cell 2 — Load PopQA and apply bands

PopQA contains 14,267 questions. Some `(s_uri, question)` pairs appear twice with
identical content (different `id`); we deduplicate those first. We then keep one
question per subject entity by shuffling with the fixed seed — this is the pool from
which new questions are drawn.

In [2]:
from datasets import load_dataset

pq = load_dataset(cfg["questions"]["dataset"], split="test").to_pandas()
print(f"PopQA loaded: {len(pq):,} rows")

# drop exact (s_uri, question) duplicates — same content, different id
n_before = len(pq)
pq = pq.drop_duplicates(["s_uri", "question"]).copy()
print(f"after dropping (s_uri,question) duplicates: {len(pq):,} rows (removed {n_before-len(pq)})")

# apply bands
pq["band"] = pq["s_pop"].apply(
    lambda x: "tail" if x < TAIL_MAX else ("head" if x > HEAD_MIN else None))
banded = pq[pq["band"].notna()].copy()

print(f"\nband counts in full PopQA (before deduplication by entity):")
print(banded["band"].value_counts().to_string())

# one question per entity: shuffle then drop_duplicates keeps a deterministic choice
pq_one = banded.sample(frac=1, random_state=SEED).drop_duplicates("s_uri", keep="first")
print(f"\nafter keeping one question per entity:")
print(pq_one["band"].value_counts().to_string())
print(f"total unique entities available: {len(pq_one):,}")

E:\shaithu\4th Year\IRP\halt-rag-repo\HALT-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo card metadata block was not found. Setting CardData to empty.


PopQA loaded: 14,267 rows
after dropping (s_uri,question) duplicates: 13,800 rows (removed 467)



band counts in full PopQA (before deduplication by entity):
band
head    8394
tail    1392

after keeping one question per entity:
band
head    7103
tail    1367
total unique entities available: 8,470


## Cell 3 — Load pilot questions, match to PopQA, and apply band rules strictly

The pilot audit (`halt_rag_audit_v5.csv`) does not carry `s_pop`, `id` or `prop`.
`halt_rag_audit_w23.csv` does carry `s_uri` (added by `02b`), so we use it to match
each pilot question back to PopQA by `(s_uri, question)`.

After matching, **band rules are applied strictly** using the current PopQA `s_pop`:

* Tail pilot questions with `s_pop >= 100` are dropped.
* Head pilot questions with `s_pop <= 500` are dropped.
* If the same subject entity appears more than once after those drops, the second
  occurrence is also dropped.

Dropped questions are replaced with new questions from the same band in Cell 4.

In [3]:
# audit_w23 has s_uri; audit_v5 has the original band and wiki_title
pilot_base = pd.read_csv(PILOT_CSV)   # 300 rows, original columns
pilot_w23  = pd.read_csv(AUDIT_W23)   # 300 rows, same order + s_uri, qid

assert len(pilot_base) == len(pilot_w23) == 300
assert (pilot_base["question"].values == pilot_w23["question"].values).all(), \
    "audit_v5 and audit_w23 are not in the same row order"

# start from audit_v5 and add s_uri from audit_w23
pilot = pilot_base[["question", "answers_list", "band", "wiki_title"]].copy()
pilot["s_uri"] = pilot_w23["s_uri"].values

# match to PopQA to get s_pop, id, prop
# join key: (s_uri, question) — unique in the deduplicated PopQA
pq_meta = pq[["s_uri", "question", "s_pop", "id", "prop"]].drop_duplicates(["s_uri", "question"])
pilot = pilot.merge(pq_meta, on=["s_uri", "question"], how="left")

n_matched = pilot["id"].notna().sum()
print(f"pilot questions matched in PopQA by (s_uri, question): {n_matched} / 300")
assert n_matched == 300, "not all pilot questions matched — investigate before continuing"

# --- apply band rules strictly ---
# a tail pilot question must have s_pop < TAIL_MAX; a head must have s_pop > HEAD_MIN
tail_ok  = (pilot["band"] == "tail") & (pilot["s_pop"] < TAIL_MAX)
head_ok  = (pilot["band"] == "head") & (pilot["s_pop"] > HEAD_MIN)
band_ok  = tail_ok | head_ok

drop_band = pilot[~band_ok].copy()
drop_band["drop_reason"] = drop_band.apply(
    lambda r: f"tail but s_pop={int(r.s_pop)} >= {TAIL_MAX}" if r.band == "tail"
              else f"head but s_pop={int(r.s_pop)} <= {HEAD_MIN}", axis=1)

# --- entity deduplication on the remaining pilot questions ---
after_band = pilot[band_ok].copy().reset_index(drop=True)
dup_mask   = after_band.duplicated("s_uri", keep="first")
drop_dup   = after_band[dup_mask].copy()
drop_dup["drop_reason"] = "duplicate entity (second occurrence)"
after_dedup = after_band[~dup_mask].copy().reset_index(drop=True)

# combine dropped
all_dropped = pd.concat([drop_band, drop_dup], ignore_index=True)

# report
n_drop_tail = int((all_dropped["band"] == "tail").sum())
n_drop_head = int((all_dropped["band"] == "head").sum())
print(f"\nPILOT QUESTIONS DROPPED: {len(all_dropped)} total  "
      f"({n_drop_head} head, {n_drop_tail} tail)")
if len(all_dropped):
    print(all_dropped[["question", "band", "wiki_title", "s_pop", "drop_reason"]].to_string(index=False))

pilot_valid = after_dedup
n_pv_head = int((pilot_valid["band"] == "head").sum())
n_pv_tail = int((pilot_valid["band"] == "tail").sum())
print(f"\npilot_valid: {len(pilot_valid)} questions ({n_pv_head} head, {n_pv_tail} tail)")
pv_tail = pilot_valid[pilot_valid["band"] == "tail"]
pv_head = pilot_valid[pilot_valid["band"] == "head"]
print(f"  tail s_pop: min {int(pv_tail['s_pop'].min())} | max {int(pv_tail['s_pop'].max())}")
print(f"  head s_pop: min {int(pv_head['s_pop'].min())} | max {int(pv_head['s_pop'].max())}")

pilot questions matched in PopQA by (s_uri, question): 300 / 300

PILOT QUESTIONS DROPPED: 5 total  (2 head, 3 tail)
                                  question band                        wiki_title  s_pop                          drop_reason
                       What genre is Gone? tail              Gone (Entwine album)    771            tail but s_pop=771 >= 100
              Who is the author of Shadow? tail                Shadow (Star Trek)    166            tail but s_pop=166 >= 100
       Who was the screenwriter for Pilot? tail Pilot (Sabrina the Teenage Witch)   1979           tail but s_pop=1979 >= 100
Who was the screenwriter for The Spoilers? head          The Spoilers (1955 film)    160            head but s_pop=160 <= 500
           Who is the mother of George VI? head                         George VI 420204 duplicate entity (second occurrence)

pilot_valid: 295 questions (148 head, 147 tail)
  tail s_pop: min 2 | max 74
  head s_pop: min 502 | max 420204


## Cell 4 — Build the new question pool

New questions must come from entities **not already in `pilot_valid`**. Within each
band we sample exactly enough questions to reach 750 — this includes replacements for
any dropped pilot questions.

In [4]:
pilot_uris = set(pilot_valid["s_uri"].dropna())
n_pilot_head = int((pilot_valid["band"] == "head").sum())
n_pilot_tail = int((pilot_valid["band"] == "tail").sum())
n_new_head   = N_PER_BAND - n_pilot_head
n_new_tail   = N_PER_BAND - n_pilot_tail

print(f"pilot_valid contributes: {n_pilot_head} head + {n_pilot_tail} tail")
print(f"new questions needed:    {n_new_head} head + {n_new_tail} tail "
      f"(includes replacements for {len(all_dropped)} dropped pilot questions)")

# pool = one question per entity, not in pilot_valid
pool = pq_one[~pq_one["s_uri"].isin(pilot_uris)]
head_pool = pool[pool["band"] == "head"]
tail_pool = pool[pool["band"] == "tail"]
print(f"\npool after removing pilot_valid entities:")
print(f"  head: {len(head_pool):,} available, need {n_new_head}")
print(f"  tail: {len(tail_pool):,} available, need {n_new_tail}")

assert len(head_pool) >= n_new_head, \
    f"not enough head questions: {len(head_pool)} available, {n_new_head} needed"
assert len(tail_pool) >= n_new_tail, \
    f"not enough tail questions: {len(tail_pool)} available, {n_new_tail} needed"

head_new = head_pool.sample(n=n_new_head, random_state=SEED)
tail_new = tail_pool.sample(n=n_new_tail, random_state=SEED)

print(f"\nsampled: {len(head_new)} new head + {len(tail_new)} new tail")
print(f"new tail s_pop: min {int(tail_new['s_pop'].min())} | "
      f"max {int(tail_new['s_pop'].max())} | median {int(tail_new['s_pop'].median())}")
print(f"new head s_pop: min {int(head_new['s_pop'].min())} | "
      f"max {int(head_new['s_pop'].max())} | median {int(head_new['s_pop'].median())}")

pilot_valid contributes: 148 head + 147 tail
new questions needed:    602 head + 603 tail (includes replacements for 5 dropped pilot questions)

pool after removing pilot_valid entities:
  head: 6,955 available, need 602
  tail: 1,220 available, need 603

sampled: 602 new head + 603 new tail
new tail s_pop: min 5 | max 99 | median 69
new head s_pop: min 502 | max 1066107 | median 3274


## Cell 5 — Combine and save

The output has exactly the columns that `02b_wiki2023_build.ipynb` expects:
`question`, `answers_list`, `band`, `wiki_title`, `s_uri`, `s_pop`, `id`, `prop`.

For new questions, `possible_answers` from PopQA is stored as `answers_list` using the
same Python `str(list(...))` repr the pilot uses, so downstream `ast.literal_eval`
calls work on both.

For `wiki_title`, new questions use `s_wiki_title` from PopQA — this is the same
field `02b` uses to look up Wikipedia pages.

In [5]:
OUT_COLS = ["question", "answers_list", "band", "wiki_title", "s_uri", "s_pop", "id", "prop"]

def to_answers_list(x):
    """Convert possible_answers string to answers_list string in the same format as the pilot."""
    try:
        lst = ast.literal_eval(x)
        return str(list(str(a) for a in lst))
    except Exception:
        return str([str(x)])

# format the new questions
def format_new(df):
    out = pd.DataFrame()
    out["question"]     = df["question"].values
    out["answers_list"] = df["possible_answers"].apply(to_answers_list).values
    out["band"]         = df["band"].values
    out["wiki_title"]   = df["s_wiki_title"].values
    out["s_uri"]        = df["s_uri"].values
    out["s_pop"]        = df["s_pop"].values
    out["id"]           = df["id"].values
    out["prop"]         = df["prop"].values
    return out

new_df = pd.concat([format_new(head_new), format_new(tail_new)], ignore_index=True)

# pilot_valid already has the right column names
pilot_out = pilot_valid[["question", "answers_list", "band", "wiki_title",
                          "s_uri", "s_pop", "id", "prop"]].copy()

result = pd.concat([pilot_out, new_df], ignore_index=True)

# --- strict band checks on all 1,500 ---
tail_viol = result[(result["band"] == "tail") & (result["s_pop"] >= TAIL_MAX)]
head_viol = result[(result["band"] == "head") & (result["s_pop"] <= HEAD_MIN)]
assert len(tail_viol) == 0, \
    f"tail band violations:\n{tail_viol[['question','s_pop']].to_string()}"
assert len(head_viol) == 0, \
    f"head band violations:\n{head_viol[['question','s_pop']].to_string()}"

# --- sanity checks ---
assert len(result) == 1500, f"expected 1500 rows, got {len(result)}"
assert result["band"].value_counts()["head"] == N_PER_BAND
assert result["band"].value_counts()["tail"] == N_PER_BAND
n_uniq_final = result["s_uri"].nunique()
assert n_uniq_final == 1500, f"expected 1500 unique entities, got {n_uniq_final}"

n_pilot_in = int(pilot_valid["question"].isin(result["question"]).sum())
new_uris   = set(new_df["s_uri"])
overlap    = new_uris & pilot_uris
assert len(overlap) == 0, f"entity overlap between new and pilot: {overlap}"

print(f"total rows:        {len(result)}")
print(f"head: {(result['band']=='head').sum()} | tail: {(result['band']=='tail').sum()}")
print(f"unique entities:   {n_uniq_final}")
print(f"pilot kept:        {n_pilot_in} / {len(pilot_valid)} (valid pilot; "
      f"{len(all_dropped)} dropped and replaced)")
print(f"new questions:     {len(new_df)}")
print(f"\ns_pop by band:")
for b in ["head", "tail"]:
    s = result[result["band"] == b]["s_pop"]
    print(f"  {b}: min {int(s.min())} | max {int(s.max())} | median {int(s.median())}")

result.to_csv(OUT_CSV, index=False)
print(f"\nsaved: {OUT_CSV}  ({len(result)} rows, {len(result.columns)} columns)")
print(f"columns: {list(result.columns)}")

total rows:        1500
head: 750 | tail: 750
unique entities:   1500
pilot kept:        295 / 295 (valid pilot; 5 dropped and replaced)
new questions:     1205

s_pop by band:
  head: min 502 | max 1066107 | median 3175
  tail: min 2 | max 99 | median 61

saved: E:\shaithu\4th Year\IRP\halt-rag-repo\HALT-RAG\data\halt_rag_audit_1500.csv  (1500 rows, 8 columns)
columns: ['question', 'answers_list', 'band', 'wiki_title', 's_uri', 's_pop', 'id', 'prop']


---

### What notebooks 02b, 03 and 04 need to change to use this file

They do not change yet — that happens before the full 1,500-question run. Notes are
kept here so nothing is forgotten.

**`02b_wiki2023_build.ipynb`** — one line changes:

```python
AUDIT_IN = data_path(cfg, "audit_1500")   # was "audit_v5"
```

That is the only change. The rest of 02b reads `s_uri` from the audit and looks up
Wikipedia pages by Wikidata ID, which works regardless of how many questions are in
the file. Output files should be renamed too (e.g. `wiki1500_cache.pkl`) so the
300-question cache is not overwritten; this needs new config keys.

**`03_signals_w23.ipynb`** and **`04_retrieval_and_hard_lift.ipynb`** — both must:

1. switch `audit_w23` → the new w1500 audit (output of 02b on the 1,500-question file);
2. switch `signals_w23` / `signals_w23_hard` → new output file names;
3. switch `corpus_embs_w23` → a new `.npy` file name, so the 300-question
   embeddings are not overwritten;
4. switch the inline `chunk_text` / `chunk_text` / corpus build to
   `src/halt_rag` (`merge_into_passages`, `dedupe`, `embedding.lift`) — this was
   already flagged as "still to do before the 1,000-question run" in both notebooks.

All of these are config changes, not logic changes.